#I want to use Paddle OCR to extract texts from the textbooks and return them as txt to work locally to make the extraction faster

In [3]:
# 1. Nuke the unstable beta versions from Colab's memory
!pip uninstall paddlepaddle-gpu paddleocr paddlex -y

# 2. Install the rock-solid Stable versions
!pip install paddlepaddle-gpu "paddleocr<3.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 758.9/758.9 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.8 MB/s eta 0:00:00
  Attempting uninstall: opt-einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0


In [5]:
!pip install  tqdm langchain-community

# 3. Install the rest of the OCR stack
!pip install paddleocr pymupdf opencv-python

In [1]:
import fitz  # PyMuPDF
import cv2
import numpy as np
from paddleocr import PaddleOCR
from tqdm import tqdm
import logging

# Suppress Paddle's excessive debug warnings for a clean output
logging.getLogger('ppocr').setLevel(logging.ERROR)

print("Initializing Stable PaddleOCR on T4 GPU...")

# Initialize Stable PaddleOCR (v2.x API)
ocr = PaddleOCR(
    use_angle_cls=True,
    lang='en',
    use_gpu=True,
    show_log=False # Keeps the terminal clean
)

def textbook_to_txt_gpu(pdf_path, output_txt_path):
    print(f"Opening {pdf_path}...")
    doc = fitz.open(pdf_path)

    with open(output_txt_path, "w", encoding="utf-8") as text_file:
        for page_num in tqdm(range(len(doc)), desc="OCR Progress"):
            page = doc[page_num]

            # High-res image extraction
            pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            img_array = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.h, pix.w, pix.n)

            if pix.n == 3:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)

            # Run stable GPU OCR
            result = ocr.ocr(img_array, cls=True)

            page_text = f"\n\n--- SOURCE_PAGE: {page_num + 1} ---\n\n"

            # Extract text safely
            if result and result[0]:
                for line in result[0]:
                    text = line[1][0]
                    page_text += text + "\n"

            text_file.write(page_text)

Initializing Stable PaddleOCR on T4 GPU...
[2026/03/02 17:49:04] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/03/02 17:49:05] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0
[2026/03/02 17:49:06] ppocr WARNING: The first GPU is used for inference by default, GPU ID: 0


In [2]:
import os # Add this import

# let's define a function to load all the pdfs from our scanned textbooks directory
def load_scanned_directory(directory_path, output_dir="extracted_texts"):
    """
    Acts as a pdf Directory loader for the scanned OCR pdfs.
    loops through the folder and processes every pdf it finds.
    """
    # Ensure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    processed_count = 0
    for filename in os.listdir(directory_path):
        if filename.lower().endswith(".pdf"): # Process only PDF files
            full_pdf_path = os.path.join(directory_path, filename)
            # Create a corresponding .txt file name in the output directory
            output_txt_filename = os.path.splitext(filename)[0] + ".txt"
            output_txt_path = os.path.join(output_dir, output_txt_filename)

            print(f"\n Starting OCR on Textbook: {filename} and saving to {output_txt_path}")

            # Use the textbook_to_txt_gpu function to process the PDF and save text
            textbook_to_txt_gpu(full_pdf_path, output_txt_path)

            processed_count += 1
    return processed_count # Return count of processed books

In [ ]:
folder = "/content/drive/MyDrive/Textbooks Paeds and OnG"
output_text_folder = "/content/extracted_textbooks" # Define an output folder

print("Starting Batch OCR Process....")
total_processed_books = load_scanned_directory(folder, output_text_folder) # Call with output_dir

print(f"\n Batch Complete! Extracted a total of {total_processed_books} textbooks.") # Correct variable and message

In [4]:
import os

pdf_path = "/content/drive/MyDrive/Textbooks Paeds and OnG/AZUBUIKE.pdf"

# 1. Define the exact FILE path, not just the folder
output_text_file = "/content/extracted_textbooks/azubuike_extracted.txt"

# 2. Safety Net: Create the 'extracted_textbooks' folder if it doesn't exist yet
os.makedirs(os.path.dirname(output_text_file), exist_ok=True)

print(f"Starting Batch OCR Process for Azubuike....")

# 3. Call the function with the specific file path
textbook_to_txt_gpu(pdf_path, output_text_file)

print(f"\n✅ Batch Complete! Extracted textbook saved to {output_text_file}.")

Starting Batch OCR Process for Azubuike....
Opening /content/drive/MyDrive/Textbooks Paeds and OnG/AZUBUIKE.pdf...


OCR Progress: 100%|██████████| 763/763 [30:14<00:00,  2.38s/it]


✅ Batch Complete! Extracted textbook saved to /content/extracted_textbooks/azubuike_extracted.txt.
